# 원본 컬럼 기준 접수→승차 대기시간 관계 분석

이 노트북은 임의 파생변수를 최대한 만들지 않고, 원본 CSV에 있는 컬럼 중심으로 `접수→승차 대기시간`과의 관계를 확인한다.

단, `접수일시`는 datetime 원본 그대로는 모델에 넣기 어렵기 때문에 최소 시간 변수만 생성한다.

- `hour`: 접수시각의 시
- `dayofweek`: 접수요일 숫자값, 월=0, 일=6
- `month`: 접수월

여기서는 `is_medical_purpose`, `is_commute_purpose`, `is_return_home_purpose` 같은 임의 목적 파생변수는 만들지 않는다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks_waiting_time" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"

RENTAL_PATH = DATA_DIR / "임차택시_대기시간_전처리.csv"
SPECIAL_PATH = DATA_DIR / "특장차_대기시간_전처리_접수유형분류.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RENTAL_PATH:", RENTAL_PATH)
print("SPECIAL_PATH:", SPECIAL_PATH)

## 1. 원본 CSV 컬럼 확인

먼저 두 CSV에 어떤 컬럼이 있는지 확인한다.

In [ ]:
rental_header = pd.read_csv(RENTAL_PATH, nrows=0).columns.tolist()
special_header = pd.read_csv(SPECIAL_PATH, nrows=0).columns.tolist()

print("임차택시 CSV 컬럼 수:", len(rental_header))
print(rental_header)

print("\n특장차 CSV 컬럼 수:", len(special_header))
print(special_header)

## 2. 바로콜 승차완료 데이터 로드

모델링 대상과 동일하게 다음 데이터만 사용한다.

- 임차택시 바로콜 승차완료
- 특장차 바로콜 승차완료

`model_group`은 원본 컬럼이 아니라, 두 CSV를 합치기 위해 만든 최소 구분 컬럼이다.

In [ ]:
def existing_cols(path, wanted_cols):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    return [col for col in wanted_cols if col in header]


rental_cols = [
    "접수일시", "예정일시", "배차일시", "승차일시", "하차일시", "취소일시",
    "출발구", "출발동", "목적구", "목적동", "이용목적", "요금", "승차거리",
    "차량구분", "장애유형",
    "접수_배차_분", "접수_승차_분", "접수_취소_분",
    "배차_취소_분", "배차_승차_분", "접수승차_날짜차이",
    "예약목적여부",
    "임차택시_바로콜여부", "임차택시_장시간예외여부", "임차택시_예약성예외여부",
    "임차택시_취소분석유형", "대기시간분석_포함여부", "대기시간분석_제외사유",
]

special_cols = [
    "접수일시", "예정일시", "배차일시", "승차일시", "하차일시", "취소일시",
    "출발구", "출발동", "목적구", "목적동", "이용목적", "요금", "승차거리",
    "차량구분", "장애유형",
    "접수_배차_분", "접수_승차_분", "접수_취소_분",
    "배차_취소_분", "배차_승차_분", "접수승차_날짜차이",
    "접수시간대", "접수시간대_HH", "접수요일", "평일주말",
    "세부이동유형", "승차거리_km", "승차거리구간",
    "특장차_접수유형", "특장차_접수유형_분류상태", "특장차_접수유형_메모",
    "접수일자", "예정일자", "취소일자", "접수시", "예정시", "예정시간",
    "취소_접수유형_후보", "_원자료_index",
    "특장차_탑승완료_필수일시존재여부", "특장차_탑승완료_시간논리정상여부",
    "심야시간사전예약_후보여부", "전일접수_후보여부", "특장차_바로콜_후보여부",
    "특장차_접수유형_후보_보완", "정기접수_목적후보여부", "정기접수_가능패턴여부",
    "동일패턴건수_보완", "예정_배차_분", "예정_승차_분",
    "특장차_접수유형_후보_최종",
]

rental = pd.read_csv(
    RENTAL_PATH,
    usecols=existing_cols(RENTAL_PATH, rental_cols),
    low_memory=False,
)

special = pd.read_csv(
    SPECIAL_PATH,
    usecols=existing_cols(SPECIAL_PATH, special_cols),
    low_memory=False,
)

print("rental shape:", rental.shape)
print("special shape:", special.shape)

In [ ]:
# 일시 컬럼 변환
for frame in [rental, special]:
    for col in ["접수일시", "예정일시", "배차일시", "승차일시", "하차일시", "취소일시", "접수일자", "예정일자", "취소일자"]:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")

# 임차택시 바로콜 승차완료
rental_model = rental[
    rental["접수일시"].notna()
    & rental["승차일시"].notna()
    & rental["임차택시_바로콜여부"].fillna(False).astype(bool)
    & rental["대기시간분석_포함여부"].fillna(False).astype(bool)
].copy()
rental_model["model_group"] = "임차택시_바로콜"

# 특장차 바로콜 승차완료
special_model = special[
    special["접수일시"].notna()
    & special["승차일시"].notna()
    & (
        special["특장차_바로콜_후보여부"].fillna(False).astype(bool)
        | special["특장차_접수유형_후보_최종"].eq("바로콜 후보")
    )
].copy()
special_model["model_group"] = "특장차_바로콜"

# 두 CSV의 컬럼을 맞춰서 결합
all_cols = sorted(set(rental_model.columns) | set(special_model.columns))
data = pd.concat(
    [
        rental_model.reindex(columns=all_cols),
        special_model.reindex(columns=all_cols),
    ],
    ignore_index=True,
)

print("combined shape:", data.shape)

## 3. 타입 정리와 최소 시간 변수 생성

임의 목적 flag는 만들지 않는다.

다만 `접수일시`에서 아래 시간 변수만 생성한다.

| 컬럼 | 의미 |
|---|---|
| `hour` | 접수시각의 시, 0~23 |
| `dayofweek` | 접수요일 숫자값, 월=0, 일=6 |
| `month` | 접수월 |

`target_min`은 최종 예측 대상인 `접수_승차_분`이다.

In [ ]:
numeric_cols = [
    "요금", "승차거리", "승차거리_km",
    "접수_배차_분", "접수_승차_분", "접수_취소_분",
    "배차_취소_분", "배차_승차_분", "접수승차_날짜차이",
    "접수시간대_HH", "접수시", "예정시",
    "예정_배차_분", "예정_승차_분", "동일패턴건수_보완",
]

for col in numeric_cols:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

# target 생성
data = data[
    data["접수_승차_분"].notna()
    & data["접수_승차_분"].ge(0)
].copy()

data["target_min"] = data["접수_승차_분"]

# 최소 시간 파생 변수만 생성
data["hour"] = data["접수일시"].dt.hour.astype("int16")
data["dayofweek"] = data["접수일시"].dt.dayofweek.astype("int16")
data["month"] = data["접수일시"].dt.month.astype("int16")

# 승차거리_km 통일: 특장차는 원본 승차거리_km, 임차택시는 승차거리 사용
if "승차거리_km" not in data.columns:
    data["승차거리_km"] = np.nan

if "승차거리" in data.columns:
    data["승차거리_km"] = data["승차거리_km"].fillna(
        pd.to_numeric(data["승차거리"], errors="coerce")
    )

# object 결측은 표시용으로 미상 처리
object_cols = data.select_dtypes(include="object").columns
for col in object_cols:
    data[col] = data[col].fillna("미상").astype(str)

print("final data shape:", data.shape)
display(data.head())

display(
    data.groupby("model_group")["target_min"]
    .agg(건수="count", 중앙값="median", 평균="mean", p90=lambda x: x.quantile(0.90))
    .round(2)
)

## 4. 컬럼 출처 정리

아래 표는 현재 `data`에 있는 컬럼이 어느 CSV에서 왔는지 확인하기 위한 표다.

- `임차택시 CSV`: `임차택시_대기시간_전처리.csv`
- `특장차 CSV`: `특장차_대기시간_전처리_접수유형분류.csv`
- `노트북 생성`: 이 노트북에서 새로 만든 최소 컬럼

In [ ]:
created_cols = ["model_group", "target_min", "hour", "dayofweek", "month"]

column_source_rows = []
for col in data.columns:
    in_rental = col in rental_header
    in_special = col in special_header

    if col in created_cols:
        source = "노트북 생성"
    elif in_rental and in_special:
        source = "임차택시 CSV + 특장차 CSV"
    elif in_rental:
        source = "임차택시 CSV"
    elif in_special:
        source = "특장차 CSV"
    else:
        source = "확인 필요"

    column_source_rows.append({
        "컬럼": col,
        "출처": source,
        "임차택시_CSV_존재": in_rental,
        "특장차_CSV_존재": in_special,
        "dtype": str(data[col].dtype),
        "고유값수": data[col].nunique(dropna=True),
    })

column_source_df = pd.DataFrame(column_source_rows)
display(column_source_df.sort_values(["출처", "컬럼"]).reset_index(drop=True))

## 5. 누수 컬럼과 제외 컬럼 지정

상관관계가 높아도 아래 컬럼들은 모델 입력으로 쓰면 안 된다.

| 구분 | 컬럼 | 이유 |
|---|---|---|
| 누수 | `접수_승차_분` | 예측 대상 자체 |
| 누수 | `접수_배차_분` | 접수 이후 배차 결과 |
| 누수 | `배차_승차_분` | 배차 이후 승차 결과 |
| 누수 | `접수_취소_분` | 취소 결과 |
| 누수 | `배차_취소_분` | 취소 결과 |
| 누수 | `예정_배차_분`, `예정_승차_분` | 예약/전일접수용 사후 결과 |
| 비추천 | `요금` | 실제 운행 이후 확정될 가능성이 큼 |
| 중복 | `접수시간대`, `접수시간대_HH`, `접수시` | `hour`와 중복 |

In [ ]:
TARGET_COL = "target_min"

LEAKAGE_COLS = [
    "접수_승차_분",
    "접수_배차_분",
    "배차_승차_분",
    "접수_취소_분",
    "배차_취소_분",
    "예정_배차_분",
    "예정_승차_분",
    "접수승차_날짜차이",
]

NOT_RECOMMENDED_COLS = [
    "요금",
    "_원자료_index",
]

DUPLICATE_TIME_COLS = [
    "접수시간대",
    "접수시간대_HH",
    "접수시",
]

RAW_DATETIME_COLS = [
    "접수일시", "예정일시", "배차일시", "승차일시", "하차일시", "취소일시",
    "접수일자", "예정일자", "취소일자",
]

HELPER_COLS = [
    "임차택시_바로콜여부",
    "임차택시_장시간예외여부",
    "임차택시_예약성예외여부",
    "임차택시_취소분석유형",
    "대기시간분석_포함여부",
    "대기시간분석_제외사유",
    "특장차_접수유형",
    "특장차_접수유형_분류상태",
    "특장차_접수유형_메모",
    "취소_접수유형_후보",
    "특장차_탑승완료_필수일시존재여부",
    "특장차_탑승완료_시간논리정상여부",
    "심야시간사전예약_후보여부",
    "전일접수_후보여부",
    "특장차_바로콜_후보여부",
    "특장차_접수유형_후보_보완",
    "특장차_접수유형_후보_최종",
    "정기접수_목적후보여부",
    "정기접수_가능패턴여부",
    "동일패턴건수_보완",
    "예약목적여부",
]


def usage_status(col):
    if col == TARGET_COL:
        return "target"
    if col in LEAKAGE_COLS:
        return "누수"
    if col in NOT_RECOMMENDED_COLS:
        return "비추천"
    if col in DUPLICATE_TIME_COLS:
        return "중복 시간"
    if col in RAW_DATETIME_COLS:
        return "원시 일시"
    if col in HELPER_COLS:
        return "전처리/분류 보조"
    return "검토 가능"

## 6. 수치형 컬럼과 접수→승차 대기시간의 상관관계

수치형 컬럼은 Pearson 상관계수로 확인한다.

그래프는 두 개를 출력한다.

1. 전체 수치형 컬럼
2. 누수/중복/비추천을 제외한 검토 가능 수치형 컬럼

In [ ]:
numeric_data = data.select_dtypes(include=[np.number]).copy()

num_corr = (
    numeric_data
    .corr(numeric_only=True)[TARGET_COL]
    .drop(labels=[TARGET_COL], errors="ignore")
    .dropna()
)

num_relation_df = pd.DataFrame({
    "컬럼": num_corr.index,
    "변수유형": "수치형",
    "고유값수": [data[col].nunique(dropna=True) for col in num_corr.index],
    "관계지표": num_corr.abs().values,
    "원상관계수": num_corr.values,
    "지표명": "abs_pearson_corr",
})
num_relation_df["사용구분"] = num_relation_df["컬럼"].map(usage_status)
num_relation_df = num_relation_df.sort_values("관계지표", ascending=False).reset_index(drop=True)

display(num_relation_df.round(4))

In [ ]:
def plot_numeric_corr(frame, title):
    plot_df = frame.sort_values("원상관계수", ascending=True).copy()
    colors = plot_df["원상관계수"].apply(lambda x: "#e15759" if x > 0 else "#4e79a7")

    plt.figure(figsize=(10, max(6, len(plot_df) * 0.35)))
    plt.barh(plot_df["컬럼"], plot_df["원상관계수"], color=colors, alpha=0.85)
    plt.axvline(0, color="black", linewidth=1)
    plt.title(title)
    plt.xlabel("Pearson 상관계수")
    plt.ylabel("컬럼")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_numeric_corr(num_relation_df, "전체 수치형 컬럼과 접수→승차 대기시간의 상관관계")

safe_num_relation_df = num_relation_df[num_relation_df["사용구분"].eq("검토 가능")].copy()
plot_numeric_corr(safe_num_relation_df, "검토 가능 수치형 컬럼과 접수→승차 대기시간의 상관관계")

## 7. 범주형 컬럼과 접수→승차 대기시간의 관계

범주형 컬럼은 Pearson 상관계수를 계산할 수 없으므로 `eta squared`를 사용한다.

여기에는 `출발구`, `출발동`, `목적구`, `목적동`, `이용목적`, `세부이동유형` 등이 포함된다.

In [ ]:
def correlation_ratio(categories, values):
    valid = pd.DataFrame({
        "category": categories,
        "value": values,
    }).dropna()

    if valid["category"].nunique() <= 1:
        return np.nan

    grand_mean = valid["value"].mean()
    total_sum = ((valid["value"] - grand_mean) ** 2).sum()

    if total_sum == 0:
        return np.nan

    between_sum = 0
    for _, group in valid.groupby("category"):
        between_sum += len(group) * ((group["value"].mean() - grand_mean) ** 2)

    return between_sum / total_sum


categorical_cols = data.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

cat_relation_rows = []
for col in categorical_cols:
    if col in RAW_DATETIME_COLS:
        continue

    nunique = data[col].nunique(dropna=True)

    # 고유값이 너무 많은 컬럼은 eta squared가 과대평가될 수 있어서 제외
    if nunique > 1000:
        continue

    cat_relation_rows.append({
        "컬럼": col,
        "변수유형": "범주형",
        "고유값수": nunique,
        "관계지표": correlation_ratio(data[col], data[TARGET_COL]),
        "원상관계수": np.nan,
        "지표명": "eta_squared",
        "사용구분": usage_status(col),
    })

cat_relation_df = (
    pd.DataFrame(cat_relation_rows)
    .dropna(subset=["관계지표"])
    .sort_values("관계지표", ascending=False)
    .reset_index(drop=True)
)

display(cat_relation_df.round(4))

In [ ]:
def plot_cat_relation(frame, title):
    plot_df = frame.sort_values("관계지표", ascending=True).copy()

    plt.figure(figsize=(10, max(6, len(plot_df) * 0.35)))
    plt.barh(plot_df["컬럼"], plot_df["관계지표"], color="#59a14f", alpha=0.85)
    plt.title(title)
    plt.xlabel("eta squared")
    plt.ylabel("컬럼")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_cat_relation(cat_relation_df, "전체 범주형 컬럼과 접수→승차 대기시간의 관계")

safe_cat_relation_df = cat_relation_df[cat_relation_df["사용구분"].eq("검토 가능")].copy()
plot_cat_relation(safe_cat_relation_df, "검토 가능 범주형 컬럼과 접수→승차 대기시간의 관계")

## 8. 전체 변수 관계지표 통합

수치형과 범주형을 한 표로 합친다.

주의: 수치형은 `|Pearson|`, 범주형은 `eta squared`라서 완전히 같은 기준의 숫자는 아니다. 모델 후보를 고르는 참고용으로만 사용한다.

In [ ]:
all_relation_df = pd.concat(
    [num_relation_df, cat_relation_df],
    ignore_index=True,
).sort_values("관계지표", ascending=False).reset_index(drop=True)

safe_relation_df = all_relation_df[all_relation_df["사용구분"].eq("검토 가능")].copy()

print("전체 변수 관계지표")
display(all_relation_df.round(4))

print("검토 가능 변수 관계지표")
display(safe_relation_df.round(4))

In [ ]:
def plot_all_relation(frame, title):
    plot_df = frame.sort_values("관계지표", ascending=True).copy()
    colors = plot_df["변수유형"].map({"수치형": "#4e79a7", "범주형": "#59a14f"}).fillna("#999999")

    plt.figure(figsize=(11, max(8, len(plot_df) * 0.35)))
    plt.barh(plot_df["컬럼"], plot_df["관계지표"], color=colors, alpha=0.85)
    plt.title(title)
    plt.xlabel("관계지표 | 수치형: |Pearson|, 범주형: eta squared")
    plt.ylabel("컬럼")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_all_relation(all_relation_df, "전체 변수와 접수→승차 대기시간의 관계")
plot_all_relation(safe_relation_df, "검토 가능 변수와 접수→승차 대기시간의 관계")

## 9. 주요 범주형 컬럼별 대기시간 요약

관계지표가 높은 범주형 변수는 실제로 어떤 값에서 대기시간이 긴지 확인한다.

In [ ]:
def summarize_target_by_category(frame, category_col, target_col=TARGET_COL, min_count=100):
    summary = (
        frame
        .groupby(category_col, dropna=False)
        .agg(
            건수=(target_col, "size"),
            평균=(target_col, "mean"),
            중앙값=(target_col, "median"),
            p75=(target_col, lambda x: x.quantile(0.75)),
            p90=(target_col, lambda x: x.quantile(0.90)),
        )
        .reset_index()
    )
    summary = summary[summary["건수"].ge(min_count)].copy()
    return summary.sort_values("p90", ascending=False)

summary_cols = [
    "model_group", "차량구분", "출발구", "목적구", "출발동", "목적동",
    "세부이동유형", "이용목적", "장애유형", "승차거리구간",
]
summary_cols = [col for col in summary_cols if col in data.columns]

for col in summary_cols:
    print(f"\n[{col}] 기준 대기시간 요약")
    display(summarize_target_by_category(data, col, min_count=100).round(2))

## 10. 원본 컬럼 기준 모델 후보

이 노트북에서는 임의 목적 flag를 만들지 않았으므로, `이용목적`은 그대로 범주형 변수로 사용한다.

현재 기준으로 검토 가능한 모델 후보는 아래와 같다.

```python
ORIGINAL_BASE_FEATURES = [
    "model_group",
    "출발구",
    "목적구",
    "세부이동유형",
    "이용목적",
    "장애유형",
    "hour",
    "dayofweek",
    "month",
    "승차거리_km",
]
```

주의:

- `접수_승차_분`, `접수_배차_분`, `배차_승차_분`은 누수라 제외한다.
- `접수시간대`, `접수시간대_HH`, `접수시`는 `hour`와 중복이라 제외한다.
- `출발동`, `목적동`은 고유값이 많아서 2차 실험 후보로 둔다.

In [ ]:
ORIGINAL_BASE_FEATURES = [
    "model_group",
    "출발구",
    "목적구",
    "세부이동유형",
    "이용목적",
    "장애유형",
    "hour",
    "dayofweek",
    "month",
    "승차거리_km",
]

ORIGINAL_BASE_FEATURES = [col for col in ORIGINAL_BASE_FEATURES if col in data.columns]

print("원본 컬럼 기준 모델 후보 피처 수:", len(ORIGINAL_BASE_FEATURES))
print(ORIGINAL_BASE_FEATURES)

display(safe_relation_df[safe_relation_df["컬럼"].isin(ORIGINAL_BASE_FEATURES)].round(4))